# 1 · Del comportamiento al modelo: inferencia

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decostruttivismo/IIC3800-2026-NCC-IA/blob/main/clase-24-agosto/notebooks/soluciones/01_inferencia_SOLUCIONES.ipynb)

**Modelos de Aprendizaje en Neurociencia Cognitiva e Inteligencia Artificial**  
Clase del 24 de agosto · Interferencia cognitiva (Stroop)

---

### Qué vamos a hacer

Tenemos los datos de un experimento Stroop: 30 participantes × 100 ensayos.
En cada ensayo hay un **tiempo de reacción** y si la respuesta fue **correcta**.

> **Los datos son simulados.** No provienen de un experimento con personas: se
> generaron con la estructura de uno. Eso no cambia nada de lo que vamos a hacer,
> porque lo que se practica aquí son las decisiones de análisis, y esas son las
> mismas. Pero conviene saberlo por dos razones. La primera es de honestidad: no
> vamos a descubrir nada sobre la cognición humana en esta sesión. La segunda es más
> interesante, y volvemos a ella al final del cuaderno 2 — los datos se generaron
> con el mismo tipo de modelo que vamos a ajustar, así que todo va a encajar
> sospechosamente bien, y saber en qué se diferenciaría un conjunto real es parte de
> lo que hay que aprender.

La pregunta de la clase es cómo se pasa de esa tabla a una afirmación como
*«la incongruencia cuesta 65 milisegundos»*, y qué hace falta creer para que esa
afirmación esté justificada.

El esquema que organiza todo es

$$Y = f(X, \theta) + \epsilon$$

| símbolo | en este experimento |
|---|---|
| $X$ | la congruencia del estímulo (y otras variables que midamos) |
| $Y$ | el tiempo de reacción, o el acierto |
| $\theta$ | los parámetros latentes: cuánto pesa cada cosa |
| $\epsilon$ | la variabilidad que el modelo no explica |

Este cuaderno recorre cuatro modelos de $f$, cada uno de los cuales arregla un
problema del anterior.

> **Cómo se ejecuta una celda:** clic en la celda y `Shift + Enter`.  
> Las celdas marcadas **✋ TU TURNO** son para que escribas tú.

## 1. Cargar los datos

El CSV se lee por URL desde el repositorio del curso: no hay que subir nada.

In [ ]:
# --- Configuración: de dónde se leen los datos -------------------------------
USUARIO = "decostruttivismo"
REPO    = "IIC3800-2026-NCC-IA"
RAMA    = "main"
CARPETA = "clase-24-agosto"
ARCHIVO = "datos/clase_24agosto_dataset.csv"

URL = f"https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/{CARPETA}/{ARCHIVO}"

import os
CANDIDATOS = [ARCHIVO,
              os.path.join("..", ARCHIVO),
              os.path.join("..", "..", ARCHIVO),
              os.path.basename(ARCHIVO)]
FUENTE = next((p for p in CANDIDATOS if os.path.exists(p)), URL)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv(FUENTE)
print("Leyendo desde:", FUENTE)
print("Filas y columnas:", data.shape)
data.head()

Las columnas son:

| columna | qué es |
|---|---|
| `subject` | identificador del participante (`S01`…`S30`) |
| `trial` | número de ensayo dentro del participante (1–100) |
| `condition` | `Congruent` o `Incongruent` |
| `RT_ms` | tiempo de reacción en milisegundos |
| `accuracy` | 1 = correcto, 0 = incorrecto |
| `age` | edad del participante en años |

Antes de modelar, conviene mirar la forma de los datos: cuántos participantes hay,
si el diseño está balanceado y si falta algo.

In [ ]:
print('Participantes:', data['subject'].nunique())
print('Ensayos por participante:', data.groupby('subject').size().unique())
print()
print(data['condition'].value_counts())
print()
print('Valores faltantes por columna:')
print(data.isna().sum())

## 2. La primera mirada: promedios por condición

El resumen más simple posible de un experimento de dos condiciones.

In [ ]:
data.groupby('condition')[['RT_ms', 'accuracy']].mean()

Deberías ver algo cercano a esto:

| condición | RT medio (ms) | proporción de aciertos |
|---|---|---|
| Congruent | 518.6 | 0.940 |
| Incongruent | 583.7 | 0.884 |

Es decir: **unos 65 ms más lento** y **unos 5.6 puntos porcentuales menos preciso**
cuando el color y la palabra se contradicen. Ese es el efecto Stroop.

Dos advertencias que valen para todo lo que sigue:

1. Una diferencia de medias no dice nada sobre si la diferencia es **fiable**. Para eso
   hace falta un modelo y una medida de incertidumbre.
2. Una media colapsa 1500 ensayos en un número. Perdemos de vista que hay 30 personas
   distintas dentro, y esa pérdida nos va a costar cara en la sección 5.

Miremos la distribución completa antes de resumirla.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

for cond, color in [('Congruent', '#4C72B0'), ('Incongruent', '#C44E52')]:
    ax[0].hist(data.loc[data['condition'] == cond, 'RT_ms'],
               bins=40, alpha=0.6, label=cond, color=color)
ax[0].set_xlabel('RT (ms)'); ax[0].set_ylabel('ensayos')
ax[0].set_title('Distribución del RT'); ax[0].legend()

medias = data.groupby('condition')['RT_ms'].mean()
ax[1].bar(medias.index, medias.values, color=['#4C72B0', '#C44E52'])
ax[1].set_ylabel('RT medio (ms)'); ax[1].set_ylim(0, 650)
ax[1].set_title('Medias por condición')

plt.tight_layout(); plt.show()

Las dos distribuciones se solapan mucho. El efecto existe **en promedio**, pero un
ensayo incongruente cualquiera puede ser perfectamente más rápido que un ensayo
congruente cualquiera. Recuerda esta imagen: es exactamente la razón por la que en el
cuaderno 02 la predicción va a resultar mucho más pobre de lo que sugiere el valor-p.

> ### ✋ TU TURNO — 1
>
> Calcula el **efecto Stroop de cada participante**: para cada `subject`, el RT medio
> incongruente menos el RT medio congruente.
>
> Preguntas a contestar mirando el resultado:
>
> - ¿Cuántos de los 30 participantes muestran el efecto en la dirección esperada?
> - ¿Cuál es el rango del efecto entre participantes?
>
> *Pista:* `data.groupby(['subject','condition'])['RT_ms'].mean().unstack()`

In [ ]:
piv = data.groupby(['subject', 'condition'])['RT_ms'].mean().unstack()
efecto = piv['Incongruent'] - piv['Congruent']

print('Participantes con efecto positivo: %d de %d' % ((efecto > 0).sum(), len(efecto)))
print('Efecto medio : %.1f ms' % efecto.mean())
print('Rango        : %.1f a %.1f ms' % (efecto.min(), efecto.max()))
print('Desv. estándar : %.1f ms' % efecto.std())

efecto.sort_values().plot(kind='bar', figsize=(11, 3), color='#C44E52')
plt.ylabel('efecto Stroop (ms)'); plt.xlabel('')
plt.title('Efecto Stroop por participante'); plt.tight_layout(); plt.show()

**Resultado.** 30 de 30 participantes muestran el efecto, con una media de 65.1 ms y
un rango de 43.6 a 85.1 ms (desviación estándar 10.5 ms).

Esto es una información distinta de la diferencia de medias, y más fuerte: el efecto
no lo produce un subgrupo extremo, está en todos. Fíjate también en la escala: la
variabilidad del efecto entre personas (±10 ms) es pequeña comparada con la
variabilidad de la velocidad basal entre personas, que veremos en la sección 5.

## 3. Modelo 1 — regresión lineal simple

Escribimos la media como un modelo. Para el ensayo $i$:

$$RT_i = \beta_0 + \beta_1 \cdot \text{Condición}_i + \epsilon_i, \qquad \text{Condición} \in \{0, 1\}$$

con `Congruent` codificado como 0. Entonces $\beta_0$ es el RT congruente medio y
$\beta_1$ es **exactamente** la diferencia entre condiciones: el efecto de interferencia.

La ganancia respecto de `groupby` no está en el número, que es el mismo, sino en que
ahora tenemos un error estándar, un intervalo de confianza y un valor-p.

In [ ]:
import statsmodels.formula.api as smf

m1 = smf.ols('RT_ms ~ C(condition)', data=data).fit()
print(m1.summary())

### Cómo leer esa salida

| en el `summary()` | valor | qué significa |
|---|---|---|
| `Intercept` | 518.6 | RT medio en congruente ($\beta_0$) |
| `C(condition)[T.Incongruent]` | +65.06 | el efecto Stroop en ms ($\beta_1$) |
| `std err` | 2.79 | incertidumbre sobre ese 65.06 |
| `P>|t|` | ~1e-110 | probabilidad de ver algo así de extremo si $\beta_1$ fuera 0 |
| `R-squared` | 0.154 | fracción de la varianza del RT que el modelo explica |

Las dos últimas filas dicen cosas muy distintas y conviene no confundirlas:

- El **valor-p** es astronómicamente pequeño. El efecto es todo lo fiable que se puede pedir.
- El **R²** es 0.15. El modelo explica el 15 % de la variación del RT; el 85 % restante
  es otra cosa.

Un efecto puede ser certísimo y minúsculo a la vez. Esta es la distinción que el
cuaderno 02 lleva hasta el final.

## 4. Modelo 2 — añadir el número de ensayo

Un experimento de 100 ensayos no es 100 repeticiones intercambiables: la gente practica,
y también se cansa. Si el RT deriva a lo largo de la sesión y esa deriva no está en el
modelo, se va a $\epsilon$ e infla la incertidumbre.

$$RT_i = \beta_0 + \beta_1 \cdot \text{Condición}_i + \beta_2 \cdot \text{Ensayo}_i + \epsilon_i$$

In [ ]:
m2 = smf.ols('RT_ms ~ C(condition) + trial', data=data).fit()
print(m2.summary().tables[1])
print('R² =', round(m2.rsquared, 4))

El coeficiente de `trial` es **−0.549 ms por ensayo**: a lo largo de los 100 ensayos
los participantes se aceleran unos 55 ms. Es práctica, no efecto Stroop, y el R² sube
de 0.154 a 0.190 solo por haberla nombrado.

Nota que $\beta_1$ casi no se mueve (65.06 → 64.67). Eso es lo esperable en un diseño
**aleatorizado**: como la condición se asignó al azar a lo largo de la sesión, no está
correlacionada con `trial`, y añadir `trial` no puede sesgar la estimación del efecto —
solo la hace más precisa. En datos observacionales esto no se cumple, y ahí añadir una
variable sí cambia las conclusiones.

> ### ✋ TU TURNO — 2
>
> Dos cosas:
>
> 1. ¿El efecto Stroop **cambia** a lo largo de la sesión? Ajusta un modelo con
>    interacción: `'RT_ms ~ C(condition) * trial'` y mira el término de interacción.
> 2. Añade `age` al modelo 2. ¿La edad predice el RT en esta muestra?
>
> En ambos casos, decide qué concluyes mirando el coeficiente **y** su valor-p.

In [ ]:
m_int = smf.ols('RT_ms ~ C(condition) * trial', data=data).fit()
print(m_int.summary().tables[1])
print()
m_edad = smf.ols('RT_ms ~ C(condition) + trial + age', data=data).fit()
print(m_edad.summary().tables[1])

**1. Interacción.** El término `C(condition)[T.Incongruent]:trial` vale +0.039 ms por
ensayo con p = 0.68. No hay evidencia de que la interferencia se atenúe (ni crezca) con
la práctica: los participantes se aceleran en las dos condiciones por igual y el coste
de la incongruencia se mantiene. Es un resultado interesante en sí: la práctica mejora
la ejecución sin reducir el conflicto.

**2. Edad.** El coeficiente es +0.28 ms por año y el OLS le asigna p = 0.26. La
conclusión —no hay efecto detectable— es correcta, pero **ese valor-p no lo es**, y
conviene ver por qué antes de seguir.

`age` es una variable **entre participantes**: no cambia de un ensayo al siguiente.
Hay 30 observaciones independientes de la edad, una por persona, no 3000. Al tratar
las 3000 filas como independientes, el OLS calcula el error estándar como si tuviera
cien veces más información sobre la edad de la que tiene. Compara:

| modelo | coef. | e.e. | p |
|---|---|---|---|
| OLS sobre 3000 filas | 0.278 | 0.248 | 0.263 |
| mixto con `groups=subject` | 0.278 | **2.025** | **0.891** |
| regresión sobre los 30 promedios | 0.278 | 2.025 | 0.892 |

El error estándar correcto es **ocho veces mayor**. Y fíjate en que las dos últimas
filas prácticamente coinciden: el coeficiente es idéntico y el error estándar
concuerda hasta el tercer decimal (2.0251 frente a 2.0255). Con un diseño balanceado,
el modelo mixto acaba haciendo lo mismo que promediar cada participante y regresar
sobre los 30 promedios, que es lo único que la edad puede sostener.

Aquí esta equivocación no cambió la conclusión porque el efecto era nulo de todos
modos. Con un
efecto moderado sí la habría cambiado, y en la dirección peligrosa: un valor-p
espectacular para algo que descansa en 30 datos.

Este es exactamente el problema de la sección siguiente, y la razón de que exista el
modelo mixto.

## 5. El problema: los ensayos no son independientes

La regresión de arriba supone que las 3000 filas son 3000 observaciones independientes.
No lo son: vienen de 30 personas, 100 filas cada una. Si un participante es lento, sus
100 filas son lentas juntas.

Cuánto importa esto es una cuestión empírica. Miremos.

In [ ]:
por_sujeto = data.groupby('subject')['RT_ms'].mean().sort_values()

por_sujeto.plot(kind='bar', figsize=(11, 3), color='#55A868')
plt.axhline(data['RT_ms'].mean(), color='k', ls='--', lw=1, label='media global')
plt.ylabel('RT medio (ms)'); plt.xlabel('')
plt.title('RT medio por participante'); plt.legend(); plt.tight_layout(); plt.show()

print('Del más rápido al más lento: %.1f a %.1f ms' % (por_sujeto.min(), por_sujeto.max()))

El participante más rápido promedia 434 ms y el más lento 665 ms: una diferencia de
**231 ms**, más de tres veces el efecto que nos interesa medir.

Esa variabilidad entre personas no es ruido experimental: es sistemática, y es el
término que aísla la descomposición de la lámina,

$$\underbrace{\text{Observado}}_{RT_{ij}} = \underbrace{\text{Señal del experimento}}_{\text{efecto fijo: condición, ensayo}} + \underbrace{\text{Señal individual}}_{\text{efecto aleatorio: } u_j} + \underbrace{\text{Ruido}}_{\epsilon_{ij}}$$

El modelo mixto añade un intercepto propio por participante:

$$RT_{ij} = \beta_0 + \beta_1 \text{Condición}_{ij} + \beta_2 \text{Ensayo}_{ij} + u_j + \epsilon_{ij}, \qquad u_j \sim \mathcal{N}(0, \sigma^2_{\text{sujeto}})$$

El subíndice $j$ es el participante, $i$ el ensayo. $u_j$ dice «este participante es
$u_j$ milisegundos más lento que la media», y se estima como una desviación con
distribución, no como 30 parámetros libres.

In [ ]:
m3 = smf.mixedlm('RT_ms ~ C(condition) + trial',
                 data=data,
                 groups=data['subject']).fit()
print(m3.summary())

In [ ]:
var_sujeto = float(m3.cov_re.iloc[0, 0])
var_resid  = float(m3.scale)
icc = var_sujeto / (var_sujeto + var_resid)

print('Varianza entre participantes : %8.1f' % var_sujeto)
print('Varianza residual            : %8.1f' % var_resid)
print('ICC                          : %8.3f' % icc)

### Qué cambió

| | OLS (m2) | mixto (m3) |
|---|---|---|
| efecto de condición | 64.67 ms | 64.67 ms |
| error estándar | 2.73 | **1.69** |

La estimación es idéntica; la **incertidumbre cae un 38 %**. Al sacar la variabilidad
entre personas del término de error, el efecto de condición se mide contra un fondo
mucho más limpio. El modelo mixto no cambió la respuesta, cambió cuánto podemos
confiar en ella.

El **ICC** (coeficiente de correlación intraclase) es 0.625: el 62 % de la varianza del
RT que no explican condición ni ensayo es **variación estable entre personas**. Solo el
38 % restante es variación ensayo a ensayo.

Guarda ese número. En el cuaderno 02 vamos a intentar predecir el RT de participantes
**que el modelo nunca vio**. Ninguno de nuestros predictores —condición, ensayo, edad—
describe a la persona, así que ese 62 % va a quedar fuera del alcance del modelo: no
porque validemos mal, sino porque no hemos medido nada individual.

> ### ✋ TU TURNO — 3
>
> Hasta aquí cada participante tiene su propia *velocidad* (intercepto), pero se le
> impone el *mismo* efecto Stroop. Ajusta un modelo que también deje variar el efecto:
>
> ```python
> m4 = smf.mixedlm('RT_ms ~ C(condition) + trial',
>                  data=data, groups=data['subject'],
>                  re_formula='~C(condition)').fit()
> ```
>
> Compara `Group Var` (variabilidad de la velocidad basal) con
> `C(condition)[T.Incongruent] Var` (variabilidad del efecto). ¿Cuál es mayor, y qué
> significa eso sobre los participantes?
>
> *Puede aparecer un aviso de convergencia; no es un error.*

In [ ]:
m4 = smf.mixedlm('RT_ms ~ C(condition) + trial',
                 data=data, groups=data['subject'],
                 re_formula='~C(condition)').fit()
print(m4.summary())

**Resultado.** La varianza del intercepto es ≈ 6045 (desviación estándar ≈ 78 ms) y la
varianza de la pendiente ≈ 16 (desviación estándar ≈ 4 ms).

La lectura: los participantes difieren **muchísimo** en lo rápidos que son y muy poco
en cuánto les cuesta la incongruencia. Es consistente con lo que viste en TU TURNO 1,
donde los 30 efectos individuales caían entre 44 y 85 ms alrededor de 65.

Es una afirmación con contenido psicológico: el mecanismo de interferencia parece
operar de forma parecida en todo el mundo, mientras que la velocidad general de
respuesta es un rasgo individual marcado.

El aviso de convergencia aparece porque estimar una estructura de covarianza completa
con 30 grupos es un problema difícil; el segundo optimizador (`lbfgs`) converge y por
eso el resultado se imprime.

## 6. La otra variable: aciertos

`accuracy` es binaria y la regresión lineal no es la herramienta adecuada, por dos
razones de distinto peso. La conocida es que nada impide a una recta predecir
probabilidades fuera de $[0,1]$ —en estos datos no llega a pasar, porque 0.88 y 0.94
están lejos de los extremos, pero ocurre en cuanto las probabilidades se acercan a 0 o
a 1, o el modelo tiene muchos predictores. La de fondo es que la regresión lineal
supone varianza constante, y en una variable 0/1 la varianza es $p(1-p)$: depende de la
propia media, y es máxima en 0.5 y mínima en los extremos. Esa es la razón por la que
los errores estándar de un OLS sobre una variable binaria no son de fiar aunque las
predicciones caigan dentro del rango.

La regresión logística modela la probabilidad a través de la escala de los **log-odds**:

$$p_{ij} = P(\text{Acierto}_{ij} = 1), \qquad \text{odds}_{ij} = \frac{p_{ij}}{1 - p_{ij}}, \qquad \text{logit}(p_{ij}) = \log \frac{p_{ij}}{1-p_{ij}} = \beta_0 + \beta_1 \text{Condición}_{ij}$$

y equivalentemente

$$p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

La razón de pasar por el logit es que estira $[0,1]$ a toda la recta real, de modo que
un modelo lineal en esa escala nunca produce una probabilidad imposible.

In [ ]:
modelo_log = smf.logit('accuracy ~ C(condition)', data=data).fit()
print(modelo_log.summary())

Los coeficientes están en log-odds y no se leen directamente. Hay dos traducciones.

In [ ]:
print('Coeficientes (log-odds):')
print(modelo_log.params)
print()

# Traducción 1: a probabilidades
nuevos = pd.DataFrame({'condition': ['Congruent', 'Incongruent']})
nuevos['prob_predicha_correcta'] = modelo_log.predict(nuevos)
print(nuevos)
print()

# Traducción 2: a odds ratios
print('Odds ratios:')
print(np.exp(modelo_log.params))

### Las tres escalas, en el mismo resultado

| escala | congruente | incongruente | efecto |
|---|---|---|---|
| log-odds | 2.752 | 2.031 | $\beta_1 = -0.721$ |
| odds | 15.67 | 7.62 | OR = 0.486 |
| probabilidad | 0.940 | 0.884 | −5.6 puntos |

Cómo se dice cada una en castellano:

- **Odds 15.67** = en congruente hay unas 15.7 respuestas correctas por cada incorrecta.
- **OR 0.486** = pasar a incongruente **reduce a la mitad** los odds de acertar.
- **Probabilidad 0.884** = de cada 100 ensayos incongruentes, unos 88 correctos.

Un aviso que se olvida a menudo: el odds ratio **no** es un cociente de probabilidades.
Los odds se dividen por la mitad, pero la probabilidad solo baja de 0.940 a 0.884. Cuando
las probabilidades son altas, un OR dramático corresponde a un cambio modesto en
probabilidad. Reportar solo el OR es una forma clásica de exagerar un efecto.

Fíjate por último en que las dos medidas **empeoran a la vez**: la incongruencia hace a
la gente más lenta *y* menos precisa. No hay compensación entre velocidad y precisión,
así que el efecto en RT no se puede explicar diciendo que los participantes se tomaron
más tiempo para ser más exactos.

> ### ✋ TU TURNO — 4
>
> 1. Añade `trial` al modelo logístico y comprueba si la precisión también mejora con
>    la práctica. Traduce el efecto a puntos porcentuales y compáralo con los 5.6
>    puntos que cuesta la incongruencia: ¿es `trial` una molestia menor o un efecto
>    del mismo orden que el que estudiamos?
> 2. Calcula a mano la probabilidad de acierto en incongruente a partir de los
>    coeficientes, con $p = 1/(1 + e^{-(\beta_0 + \beta_1)})$, y verifica que coincide
>    con lo que devolvió `.predict()`.

In [ ]:
# 1. Añadir trial
modelo_log2 = smf.logit('accuracy ~ C(condition) + trial', data=data).fit(disp=0)
print(modelo_log2.summary().tables[1])
print()

# 2. Reconstruir la probabilidad a mano
b0, b1 = modelo_log.params['Intercept'], modelo_log.params['C(condition)[T.Incongruent]']
p_incong = 1 / (1 + np.exp(-(b0 + b1)))
p_cong   = 1 / (1 + np.exp(-b0))
print('A mano   -> congruente %.4f | incongruente %.4f' % (p_cong, p_incong))
print('predict() ->', modelo_log.predict(pd.DataFrame(
      {'condition': ['Congruent', 'Incongruent']})).round(4).tolist())

**1.** El coeficiente de `trial` es +0.0048 log-odds por ensayo, con p = 0.033.
Acumulado sobre los 100 ensayos son 0.48 log-odds, que traducidos a probabilidad
significan pasar de 0.892 a 0.930 promediando las dos condiciones: **3.8 puntos
porcentuales**. En los datos crudos se ve igual: 0.888 de acierto en los primeros 25
ensayos y 0.923 en los últimos 25.

Puesto en escala, no es un efecto menor. La incongruencia cuesta 5.6 puntos de
precisión; la práctica recupera 3.8, o sea dos tercios de lo que estudiamos. En el RT
pasaba algo parecido: 55 ms de práctica frente a 65 ms de interferencia.

La moraleja no es «ojo con los efectos irrelevantes» sino la contraria: `trial` no es
una variable de higiene que se controla por costumbre, es un efecto del mismo orden
que el de interés. Omitirlo lo manda entero a $\epsilon$ y ensucia la estimación de
todo lo demás.

Queda pendiente un problema que este modelo tampoco resuelve: los ensayos siguen
agrupados por participante. Hay dos salidas, y no son la misma. La versión logística
del modelo mixto es `BinomialBayesMixedGLM`, y estima el efecto **condicional al**
**participante**: cuánto le cuesta la incongruencia a una persona dada. Una GEE
(`smf.gee(..., cov_struct=Exchangeable())`) estima el efecto **marginal**, promediado
sobre la población, y corrige los errores estándar por el agrupamiento. En regresión
logística los dos coeficientes **no coinciden**: el marginal queda atenuado. Cuál
usar depende de qué se quiera afirmar.

**2.** Las dos vías dan 0.9400 y 0.8840, idénticas a las proporciones muestrales: una
regresión logística con un único predictor binario reproduce exactamente las medias
de cada grupo. `.predict()` de un `logit` devuelve probabilidades, no log-odds:
aplica la función logística por dentro.

---

## Resumen del cuaderno 1

| paso | qué añadió | qué costó |
|---|---|---|
| medias por condición | el efecto existe: 65 ms | ninguna medida de incertidumbre |
| OLS `~ C(condition)` | error estándar, p, R² = 0.15 | supone ensayos independientes |
| OLS `+ trial` | la práctica vale −0.55 ms/ensayo | sigue suponiendo independencia |
| mixto (`mixedlm`, `groups=subject`) | error estándar 38 % menor; ICC = 0.62 | — |
| logístico | efecto en aciertos: OR = 0.49 | los odds no son probabilidades |

Y los dos números que hay que llevarse a la segunda mitad de la clase:

- El efecto de condición tiene $p \approx 10^{-110}$.
- El modelo explica el **15 %** de la varianza del RT.

Todo lo anterior es **inferencia**: preguntamos si un efecto es compatible con el azar
bajo un modelo dado. La otra pregunta —en qué medida el modelo generaliza a
observaciones nuevas— es **predicción**, y no se responde con nada de lo que hicimos aquí.

→ Continúa en **`02_prediccion.ipynb`**.